In [ ]:


%pip install youtube-transcript-api transformers torch --quiet

print("All libraries installed successfully!")


In [ ]:

import re          
import textwrap    
import youtube_transcript_api
import transformers
print("Both imports work")


from youtube_transcript_api import (
    YouTubeTranscriptApi,     
    NoTranscriptFound,         
    TranscriptsDisabled,     
)


from transformers import pipeline


print("All imports successful!")


In [ ]:
def extract_video_id(url: str) -> str:
    """
    Parse the 11-character YouTube video ID from any common URL format.
    Raises ValueError with a friendly message if the URL is unrecognised.
    """
    
    patterns = [
        r"(?:v=)([A-Za-z0-9_\-]{11})",      
        r"youtu\.be/([A-Za-z0-9_\-]{11})",  
        r"shorts/([A-Za-z0-9_\-]{11})",      
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)   # group(1) = first capture group = the ID

    raise ValueError(
        f"Could not find a YouTube video ID in: {url}\n"
        "Make sure you're using a standard YouTube URL."
    )

#sample --
test_urls = [
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://youtu.be/dQw4w9WgXcQ",
    "https://youtube.com/shorts/dQw4w9WgXcQ",
]
for u in test_urls:
    vid = extract_video_id(u)
    print(f"  URL: {u[:50]:<50}  →  ID: {vid}")

print("\nextract_video_id() works correctly!")


In [ ]:
def fetch_transcript(video_id: str) -> tuple[str, list]:
    """
    Download the transcript for a YouTube video.

    Returns:
        full_text (str)  — entire transcript as one string
        segments  (list) — raw caption entries with start times
                           useful for timestamps (bonus section)

    Raises:
        ValueError — with a user-friendly message on failure
    """
    print(f"Fetching transcript for video ID: {video_id} ...")

    try:
        # Try English first; fall back to any available language
        try:
            segments = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
        except NoTranscriptFound:
            print("English transcript not found — trying other languages...")
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            available = list(transcript_list)
            if not available:
                raise ValueError("No transcripts found for this video.")
            transcript = available[0].fetch()
            segments = transcript

        # Join all text segments into one continuous string
        full_text = " ".join(seg["text"] for seg in segments)

        # Clean up: collapse multiple spaces / newlines into one space
        full_text = re.sub(r"\s+", " ", full_text).strip()

        word_count = len(full_text.split())
        print(f"✅ Transcript fetched! ({word_count:,} words across {len(segments):,} segments)")
        return full_text, segments

    except TranscriptsDisabled:
        raise ValueError(
            "Transcripts are disabled for this video.\n"
            "Try a video that has captions turned on."
        )
    except Exception as e:
        raise ValueError(f"❌ Could not fetch transcript: {e}")


In [ ]:
def preprocess_text(text: str) -> str:
    """
    Clean raw transcript text before summarization.

    Operations performed (in order):
    1. Strip speaker annotations like [Music], [Laughter], [Applause]
    2. Remove URLs (transcripts sometimes contain them)
    3. Collapse multiple spaces → single space
    4. Strip leading/trailing whitespace
    """
    
    text = re.sub(r"\[.*?\]", "", text)

    text = re.sub(r"https?://\S+", "", text)

   
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


# ── Demo the cleaning on a noisy example ───────────────────────────────────
sample_noisy = "[Music] Hello everyone  [Applause]  and welcome  to   today's video! [Laughter]"
sample_clean  = preprocess_text(sample_noisy)

print("Before:", sample_noisy)
print("After: ", sample_clean)
print("\npreprocess_text() works correctly!")


In [ ]:
# ── Tunable constants ───────────────────────────────────────────────────────
MAX_WORDS_PER_CHUNK = 700   # Safe word limit per BART pass  (~900 tokens)
CHUNK_OVERLAP       = 50    # Words of overlap between consecutive chunks


def chunk_text(text: str, max_words: int = MAX_WORDS_PER_CHUNK,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Split text into overlapping word-based chunks.

    Args:
        text      — the full transcript string
        max_words — maximum words per chunk
        overlap   — how many words the next chunk re-reads from the previous one

    Returns:
        List of chunk strings. If text is short enough, returns [text].
    """
    words = text.split()
    total = len(words)

    if total <= max_words:
        print(f"Text is short ({total} words) — no chunking needed.")
        return [text]

    chunks = []
    start = 0
    while start < total:
        end = min(start + max_words, total)
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == total:
            break
        start += max_words - overlap   # Slide forward, keeping overlap

    print(f"Split {total:,} words into {len(chunks)} chunks "
          f"(~{max_words} words each, {overlap}-word overlap)")
    return chunks


# ── Visual demonstration with a tiny example ────────────────────────────────
demo_words = list(range(1, 16))            # [1, 2, 3, ... 15]
demo_text  = " ".join(str(w) for w in demo_words)
demo_chunks = chunk_text(demo_text, max_words=6, overlap=2)

print("\nDemo (max_words=6, overlap=2):")
for i, c in enumerate(demo_chunks, 1):
    print(f"  Chunk {i}: [{c}]")
print("\nchunk_text() works correctly!")


In [ ]:
# ── Load the model (cached after first download) ────────────────────────────
print("⏳ Loading BART model (first run downloads ~1.6 GB) ...")

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    # device=0 if you have a GPU — CPU is fine for short videos
)

print("Model loaded and ready!")


In [ ]:
def summarize_text(text: str, max_len: int = 150, min_len: int = 40) -> str:
    """
    Summarize a single chunk of text with BART.

    Args:
        text    — input text (must be ≤ 700 words / ~900 tokens)
        max_len — maximum tokens in the generated summary
        min_len — minimum tokens (prevents one-word answers)

    Returns:
        Summary string
    """
    result = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False,    # False = deterministic beam search (more factual)
    )
    return result[0]["summary_text"]


def summarize_transcript(transcript: str) -> tuple[str, list[str]]:
    """
    Full Map-Reduce summarization pipeline.

    1. Chunk the transcript
    2. MAP:    summarize each chunk independently
    3. REDUCE: combine chunk summaries → final summary

    Returns:
        final_summary  (str)       — polished paragraph
        chunk_summaries (list[str]) — per-chunk summaries (used for bullet points)
    """
    print("\nStarting summarization pipeline ...")

    chunks = chunk_text(transcript)
    n = len(chunks)

    # ── MAP phase ────────────────────────────────────────────────────────────
    chunk_summaries = []
    for i, chunk in enumerate(chunks, 1):
        print(f"Summarizing chunk {i}/{n} ...")
        summary = summarize_text(chunk)
        chunk_summaries.append(summary)
        print(f"     → {summary[:80]}...")

    # ── REDUCE phase ─────────────────────────────────────────────────────────
    if n == 1:
        # Short video — single chunk summary IS the final summary
        final_summary = chunk_summaries[0]
        print("\nSingle chunk — no reduce step needed.")
    else:
        # Combine all chunk summaries and summarize once more
        print(f"\nCombining {n} chunk summaries into final summary ...")
        combined = " ".join(chunk_summaries)
        final_summary = summarize_text(combined, max_len=250, min_len=80)

    print("\nSummarization complete!")
    return final_summary, chunk_summaries


In [ ]:
def extract_bullet_points(chunk_summaries: list[str],
                          max_points: int = 8) -> list[str]:
    """
    Turn per-chunk summaries into a clean bullet-point list.

    Splits each summary on sentence boundaries (.!?) and filters out
    very short fragments. Caps the result at max_points items.
    """
    points = []

    for summary in chunk_summaries:
        # Split on sentence-ending punctuation followed by whitespace
        sentences = re.split(r"(?<=[.!?])\s+", summary.strip())
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) < 20:      # Skip sentence fragments
                continue
            if not sentence.endswith((".", "!", "?")):
                sentence += "."         # Ensure clean ending
            points.append(sentence)

    # De-duplicate (chunk overlap can cause repeated sentences)
    seen = set()
    unique_points = []
    for p in points:
        if p not in seen:
            seen.add(p)
            unique_points.append(p)

    return unique_points[:max_points]


In [ ]:
def display_results(original_text: str, final_summary: str,
                    bullet_points: list[str], video_url: str = "") -> None:
    """
    Print a formatted summary report to the notebook output.
    """
    original_words = len(original_text.split())
    summary_words  = len(final_summary.split())
    compression    = round((1 - summary_words / original_words) * 100, 1)

    # ── Header ───────────────────────────────────────────────────────────────
    print("=" * 65)
    print("YouTube Video Summary Report")
    if video_url:
        print(f"  🔗 {video_url}")
    print("=" * 65)

    # ── Stats ────────────────────────────────────────────────────────────────
    print(f"\nTEXT STATISTICS")
    print(f"   Original transcript : {original_words:>6,} words")
    print(f"   Summary             : {summary_words:>6,} words")
    print(f"   Compression         : {compression:>5}%  (kept {100 - compression:.1f}% of length)")

    # ── Summary paragraph ────────────────────────────────────────────────────
    print(f"\n{'─' * 65}")
    print("SUMMARY\n")
    # Wrap long lines at 65 characters for readability
    wrapped = textwrap.fill(final_summary, width=65)
    print(wrapped)

    # ── Bullet points ─────────────────────────────────────────────────────────
    print(f"\n{'─' * 65}")
    print(f"KEY INSIGHTS ({len(bullet_points)} points)\n")
    for i, point in enumerate(bullet_points, 1):
        # Indent continuation lines so they align under the first word
        wrapped_point = textwrap.fill(point, width=61,
                                      initial_indent=f"  {i}. ",
                                      subsequent_indent="     ")
        print(wrapped_point)

    print(f"\n{'=' * 65}")
    print(" Done! Paste a new URL in the next cell to summarize another video.")
    print("=" * 65)


In [ ]:
# provide in url for the one below to get the summary of the desired video 

VIDEO_URL = "https://www.youtube.com/watch?v=aircAruvnKk" # in here substitue (must be caption enabled video)

# ── Pipeline ─────────────────────────────────────────────────────────────────
print("YouTube AI Summarizer — Starting Pipeline")
print("─" * 55)

# Step 1: Extract video ID
video_id = extract_video_id(VIDEO_URL)
print(f"Video ID: {video_id}")

# Step 2: Fetch transcript
transcript_raw, segments = fetch_transcript(video_id)

# Step 3: Preprocess
transcript_clean = preprocess_text(transcript_raw)
print(f"Cleaned transcript: {len(transcript_clean.split()):,} words")

# Step 4: Summarize (Map-Reduce)
final_summary, chunk_summaries = summarize_transcript(transcript_clean)

# Step 5: Extract bullet points
bullet_points = extract_bullet_points(chunk_summaries)

# Step 6: Display clean report
print()
display_results(transcript_clean, final_summary, bullet_points, VIDEO_URL)


In [ ]:
def format_timestamp(seconds: float) -> str:
    """Convert raw seconds to MM:SS display string."""    mins, secs = divmod(int(seconds), 60)
    return f"{mins:02d}:{secs:02d}"


def add_timestamps_to_points(bullet_points: list[str],
                               segments: list[dict]) -> list[tuple[str, str]]:
    """
    Pair each bullet point with an estimated timestamp.

    We divide the segment list proportionally across bullet points.
    This is an approximation — good enough for navigation purposes.

    Returns:
        List of (timestamp_str, point_text) tuples
    """
    n = len(bullet_points)
    if not segments or n == 0:
        return [("--:--", p) for p in bullet_points]

    result = []
    for i, point in enumerate(bullet_points):
        # Map this point's position (0..n-1) to a segment index
        seg_index = int((i / n) * len(segments))
        seg_index = min(seg_index, len(segments) - 1)
        ts = format_timestamp(segments[seg_index]["start"])
        result.append((ts, point))
    return result


# ── Display timestamped bullet points ────────────────────────────────────────
print("\nKEY INSIGHTS WITH TIMESTAMPS\n")
print("─" * 55)

timestamped = add_timestamps_to_points(bullet_points, segments)
for ts, point in timestamped:
    wrapped = textwrap.fill(point, width=48,
                             initial_indent="",
                             subsequent_indent="         ")
    print(f"  [{ts}]  {wrapped}")
    print()

print("─" * 55)
print("Click a timestamp in YouTube to jump to that point.")
